In [ ]:
!pip install imblearn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier  # Ini model ANN kita
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE           # Ini metode balancing kita
import warnings

# Mengabaikan peringatan agar output bersih
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Replace 'path/to/your/file.csv' with the actual path to your file in Google Drive
file_path = '/content/drive/MyDrive/data-bank.csv'

try:
    df = pd.read_csv(file_path)
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490


In [ ]:
# Muat data Anda
file_path = '/content/drive/MyDrive/data-bank.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Data {file_path} berhasil dimuat. Shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File {file_path} tidak ditemukan.")
    # exit() # Hentikan jika file tidak ada

# Bersihkan nama kolom (menghapus spasi di awal/akhir)
df.columns = df.columns.str.strip()
target = 'Bankrupt?'

Data /content/drive/MyDrive/data-bank.csv berhasil dimuat. Shape: (6819, 96)


In [ ]:
# 1. Definisikan X (fitur) dan y (target)
X = df.drop(target, axis=1)
y = df[target]

# 2. Bagi data menjadi latih dan tes
# stratify=y penting agar proporsi bangkrut di data tes sama
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Data dibagi: {X_train.shape} (Train), {X_test.shape} (Test)")

# 3. Scaling Data (WAJIB UNTUK ANN)
scaler = StandardScaler()

# Fit scaler HANYA pada data latih
X_train_scaled = scaler.fit_transform(X_train)

# Terapkan scaler yang sama ke data tes
X_test_scaled = scaler.transform(X_test)

print("Data selesai di-scaling menggunakan StandardScaler.")

Data dibagi: (4773, 95) (Train), (2046, 95) (Test)
Data selesai di-scaling menggunakan StandardScaler.


In [ ]:
print("\nDistribusi data latih SEBELUM balancing:")
print(y_train.value_counts())

# Inisialisasi SMOTE
smote = SMOTE(random_state=42)

# Terapkan SMOTE HANYA pada data latih (X_train_scaled dan y_train)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("\nDistribusi data latih SETELAH balancing (SMOTE):")
print(pd.Series(y_train_balanced).value_counts())


Distribusi data latih SEBELUM balancing:
Bankrupt?
0    4619
1     154
Name: count, dtype: int64

Distribusi data latih SETELAH balancing (SMOTE):
Bankrupt?
0    4619
1    4619
Name: count, dtype: int64


In [ ]:
# --- Sel 6 (Tuning Opsi 3: Regularisasi Kuat) ---

print("\n--- Melatih Model ANN (Tuning 3: Alpha=0.01) ---")

# Kita tingkatkan alpha 10x lipat lagi
ann_model_tuned3 = MLPClassifier(
    hidden_layer_sizes=(100,),  # <-- Tetap 1 layer
    alpha=0.01,                 # <-- PERUBAHAN DI SINI: alpha 10x lebih kuat
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

# Latih model baru
ann_model_tuned3.fit(X_train_balanced, y_train_balanced)

print("Model ANN (Tuning 3) selesai dilatih.")


--- Melatih Model ANN (Tuning 3: Alpha=0.01) ---
Model ANN (Tuning 3) selesai dilatih.


In [ ]:
# --- Sel 8 (Opsional): Tuning Threshold pada Model Baseline ---
# Pastikan Anda sudah menjalankan Sel 6 dan 7 (untuk model 'ann_model' baseline)

print("\n--- Tuning Threshold pada Model Baseline ---")

# 1. Dapatkan PROBABILITAS dari model baseline
y_proba_ann = ann_model.predict_proba(X_test_scaled)
y_proba_bangkrut = y_proba_ann[:, 1] # Ambil probabilitas untuk kelas 1

# 2. Coba threshold baru yang lebih TINGGI (standarnya 0.5)
# Kita ingin lebih yakin sebelum memprediksi 1, untuk menaikkan Precision
NEW_THRESHOLD = 0.7

# 3. Buat prediksi baru berdasarkan threshold
y_pred_tuned_thresh = (y_proba_bangkrut > NEW_THRESHOLD).astype(int)

# 4. Evaluasi hasilnya
print(f"\n--- Hasil Evaluasi (dengan Threshold = {NEW_THRESHOLD}) ---")
print(classification_report(y_test, y_pred_tuned_thresh, target_names=['Sehat (0)', 'Bangkrut (1)']))


--- Tuning Threshold pada Model Baseline ---

--- Hasil Evaluasi (dengan Threshold = 0.7) ---
              precision    recall  f1-score   support

   Sehat (0)       0.98      0.98      0.98      1980
Bangkrut (1)       0.36      0.38      0.37        66

    accuracy                           0.96      2046
   macro avg       0.67      0.68      0.67      2046
weighted avg       0.96      0.96      0.96      2046

